### 1. INSTALAR LIBRERIAS NECESARIAS

In [ ]:
!pip install optuna
!pip install scikit-plot

### 2. IMPORTACIÓN DE LIBRERÍAS

In [9]:
# Librerias generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import multiprocessing
import os
import math as mt
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                           classification_report, roc_curve, auc, f1_score,
                           precision_score, recall_score)
from sklearn.inspection import permutation_importance

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

# Imbalanced-learn
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as Pipeline_imb
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Optuna
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

# Google Colab, conexion drive
from google.colab import drive

# Configuraciones iniciales
warnings.filterwarnings('ignore')
np.random.seed(123)
plt.style.use('default')
sns.set_palette("husl")

### 3. CARGA DE DATOS

In [10]:
print("PASO 1: CARGA DE DATOSs")
# Montar Google Drive
drive.mount('/content/drive')
# Definir ruta base
path_base = '/content/drive/MyDrive/Colab Notebooks'

# Cargar dataset
dataset = pd.read_csv(f'{path_base}/Data_Cloro_Residual.csv', sep=',')

print(f"\n Dataset cargado exitosamente!")
print(f"   Dimensiones: {dataset.shape}")
print(f"   Filas: {dataset.shape[0]:,}")
print(f"   Columnas: {dataset.shape[1]}")

print("\nPrimeras 3 filas:")
display(dataset.head(3))

print("\nÚltimas 3 filas:")
display(dataset.tail(3))

print("\nInformación del dataset:")
print(dataset.info())

print("\nEstadísticas descriptivas:")
display(dataset.describe())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 Dataset cargado exitosamente!
   Dimensiones: (18855, 38)
   Filas: 18,855
   Columnas: 38

Primeras 3 filas:


,CPCSD,p214_Recibe_Cuota_Familiar_SI,p203_Flag_Menos2anios_Eleccion,p206_tiene_mujer_lider,p206_Flag_Miembros_SI_SecundariaSuperior,p206_Operarios_Con_Incentivos_Pagos,p206_Operarios_Gasfiteros,p207c_Cuenta_ConFondos_SI,p207c1_Registro_Cloro_Residual_SI,p207c1_Doc_Estatutos_OrgJass_SI,...,p230_Flag_Capa_Gestion,p230_Flag_Capa_Operacion,p329_Tipo_de_Fuente_Superficial,p2076_Costos_ServSaneamiento_XCuotaFamiliar_SI,p207c_Documentos_Tecnicos,p228_Apoyo_Tecnico,p228_Apoyo_Infraestructura,p207c_Documentos_Financieros,CapaOperacion_x_Gasfitero,Cloro_Adecuado
0,Compacta,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Compacta,0,1,0,1,0,1,0,0,1,...,0,0,0,0,0,0,0,2,0,1
2,Compacta,0,1,1,0,0,1,0,1,1,...,1,1,0,0,3,3,1,3,1,1



Últimas 3 filas:


,CPCSD,p214_Recibe_Cuota_Familiar_SI,p203_Flag_Menos2anios_Eleccion,p206_tiene_mujer_lider,p206_Flag_Miembros_SI_SecundariaSuperior,p206_Operarios_Con_Incentivos_Pagos,p206_Operarios_Gasfiteros,p207c_Cuenta_ConFondos_SI,p207c1_Registro_Cloro_Residual_SI,p207c1_Doc_Estatutos_OrgJass_SI,...,p230_Flag_Capa_Gestion,p230_Flag_Capa_Operacion,p329_Tipo_de_Fuente_Superficial,p2076_Costos_ServSaneamiento_XCuotaFamiliar_SI,p207c_Documentos_Tecnicos,p228_Apoyo_Tecnico,p228_Apoyo_Infraestructura,p207c_Documentos_Financieros,CapaOperacion_x_Gasfitero,Cloro_Adecuado
18852,Compacta,1,1,0,1,1,1,0,0,1,...,0,0,1,0,0,0,0,0,0,0
18853,Mixto,1,1,1,1,0,1,0,0,1,...,1,1,0,1,2,3,1,1,1,0
18854,Vacio,1,1,0,0,0,1,1,0,1,...,1,1,0,0,2,2,0,0,1,0



Información del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18855 entries, 0 to 18854
Data columns (total 38 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   CPCSD                                           18855 non-null  object 
 1   p214_Recibe_Cuota_Familiar_SI                   18855 non-null  int64  
 2   p203_Flag_Menos2anios_Eleccion                  18855 non-null  int64  
 3   p206_tiene_mujer_lider                          18855 non-null  int64  
 4   p206_Flag_Miembros_SI_SecundariaSuperior        18855 non-null  int64  
 5   p206_Operarios_Con_Incentivos_Pagos             18855 non-null  int64  
 6   p206_Operarios_Gasfiteros                       18855 non-null  int64  
 7   p207c_Cuenta_ConFondos_SI                       18855 non-null  int64  
 8   p207c1_Registro_Cloro_Residual_SI               18855 non-null  int64  
 9   p207c1_Doc_Es

,p214_Recibe_Cuota_Familiar_SI,p203_Flag_Menos2anios_Eleccion,p206_tiene_mujer_lider,p206_Flag_Miembros_SI_SecundariaSuperior,p206_Operarios_Con_Incentivos_Pagos,p206_Operarios_Gasfiteros,p207c_Cuenta_ConFondos_SI,p207c1_Registro_Cloro_Residual_SI,p207c1_Doc_Estatutos_OrgJass_SI,p217_220_Salud_Financiera_Cuotas_Porc,...,p230_Flag_Capa_Gestion,p230_Flag_Capa_Operacion,p329_Tipo_de_Fuente_Superficial,p2076_Costos_ServSaneamiento_XCuotaFamiliar_SI,p207c_Documentos_Tecnicos,p228_Apoyo_Tecnico,p228_Apoyo_Infraestructura,p207c_Documentos_Financieros,CapaOperacion_x_Gasfitero,Cloro_Adecuado
count,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,...,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000,18855.000000
mean,0.833201,0.396606,0.431451,0.679501,0.272713,0.721241,0.342721,0.376823,0.903580,0.104017,...,0.755184,0.864545,0.091912,0.564041,1.089737,1.969398,0.639883,1.928348,0.644657,0.349615
std,0.372806,0.489206,0.495292,0.466681,0.445366,0.448401,0.474632,0.484603,0.295174,0.182226,...,0.429989,0.342218,0.288909,0.495895,1.095139,1.238233,0.733317,1.497925,0.478630,0.476861
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,...,1.000000,1.000000,0.000000,1.000000,1.000000,3.000000,0.000000,2.000000,1.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.136364,...,1.000000,1.000000,0.000000,1.000000,2.000000,3.000000,1.000000,3.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,3.000000,3.000000,2.000000,5.000000,1.000000,1.000000


### 4. ANÁLISIS EXPLORATORIO DE DATOS (EDA)

In [17]:
print("PASO 2: EDA")
# Verificar valores nulos
print("\nValores nulos por columna:")
nulos = dataset.isnull().sum()
if any(nulos > 0):
    print(nulos[nulos > 0])
else:
    print("No hay valores nulos")

# Análisis de la variable objetivo
print("\nDistribución de la variable objetivo 'Cloro_Adecuado':")
target_dist = dataset['Cloro_Adecuado'].value_counts()
target_dist_perc = dataset['Cloro_Adecuado'].value_counts(normalize=True) * 100
for i, (valor, count) in enumerate(target_dist.items()):
    print(f"   Clase {valor}: {count} registros ({target_dist_perc.iloc[i]:.2f}%)")

PASO 2: EDA

Valores nulos por columna:
No hay valores nulos

Distribución de la variable objetivo 'Cloro_Adecuado':
   Clase 0: 12263 registros (65.04%)
   Clase 1: 6592 registros (34.96%)


### 5. DIVISIÓN DE DATOS (70% ENTRENAMIENTO, 15% VALIDACIÓN, 15% PRUEBA)

In [12]:
print("PASO 3: DIVISIÓN DE DATOS")

# Primera división: 85% para entrenamiento+validación, 15% para prueba final
X_temp, X_final, y_temp, y_final = train_test_split(
    dataset.drop('Cloro_Adecuado', axis=1),
    dataset['Cloro_Adecuado'],
    test_size=0.15,
    random_state=123,
    stratify=dataset['Cloro_Adecuado'],
    shuffle=True
)

# Segunda división: 70% entrenamiento, 15% validación (del 85% restante)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,  # 15% del total / 85% = ~0.176
    random_state=123,
    stratify=y_temp,
    shuffle=True
)

print(f"\nDivisión completada:")
print(f"   Entrenamiento (70%): {len(X_train):,} registros")
print(f"   Validación (15%): {len(X_val):,} registros")
print(f"   Prueba Final (15%): {len(X_final):,} registros")

PASO 3: DIVISIÓN DE DATOS

División completada:
   Entrenamiento (70%): 13,205 registros
   Validación (15%): 2,821 registros
   Prueba Final (15%): 2,829 registros


### 6. PREPROCESAMIENTO

In [14]:
print("PASO 4: CONFIGURACIÓN DEL PREPROCESAMIENTO")
# Identificar columnas categóricas y numéricas
categorical_columns = X_train.select_dtypes(include='object').columns.tolist()
numerical_columns = X_train.select_dtypes(include='number').columns.tolist()

print(f"\nVariables categóricas ({len(categorical_columns)}):")
for col in categorical_columns[:5]:
    print(f"  - {col}")

print(f"\nVariables numéricas ({len(numerical_columns)}):")
for col in numerical_columns[:5]:
    print(f"  - {col}")

# Crear preprocesador
prep = ColumnTransformer([
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns),
    ('scaler', StandardScaler(), numerical_columns)
], remainder='drop')

PASO 4: CONFIGURACIÓN DEL PREPROCESAMIENTO

Variables categóricas (3):
  - CPCSD
  - p213_Asociados_Inscritos_Cat2
  - p216_cuotapromedio_Cat

Variables numéricas (34):
  - p214_Recibe_Cuota_Familiar_SI
  - p203_Flag_Menos2anios_Eleccion
  - p206_tiene_mujer_lider
  - p206_Flag_Miembros_SI_SecundariaSuperior
  - p206_Operarios_Con_Incentivos_Pagos


### 7. CONFIGURACIÓN DE VALIDACIÓN CRUZADA

In [15]:
print("PASO 5: CONFIGURACIÓN DE VALIDACIÓN CRUZADA")

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=123)
print(f"Validación cruzada: 5 folds, 2 repeticiones")

PASO 5: CONFIGURACIÓN DE VALIDACIÓN CRUZADA
Validación cruzada: 5 folds, 2 repeticiones


### 8. OPTIMIZACIÓN CON OPTUNA

In [18]:
print("PASO 6: FUNCION MODELO CON OPTUNA")

def optimize_model(model_name, X_train, y_train, cv, n_trials=30):
    """
    Función para optimizar hiperparámetros con manejo de errores
    """
    print(f"Optimizando: {model_name}")

    def create_pipeline(trial):
        """Crea pipeline con validación de parámetros"""
        # Estrategia de balanceo adaptativa
        sampling_strategy = trial.suggest_float('sampling_strategy', 0.5, 0.9)

        # Seleccionar método de balanceo según el modelo
        if model_name in ['LogisticRegression', 'DecisionTree']:
            # Para modelos simples, usar solo SMOTE
            sampler = SMOTE(sampling_strategy=sampling_strategy, random_state=123)
        else:
            # Para modelos más complejos, usar SMOTETomek
            try:
                sampler = SMOTETomek(sampling_strategy=sampling_strategy, random_state=123)
            except:
                sampler = SMOTE(sampling_strategy=sampling_strategy, random_state=123)

        # Configurar parámetros según el modelo
        if model_name == 'LogisticRegression':
            # Parámetros válidos para LogisticRegression
            penalty = trial.suggest_categorical('penalty', ['l2'])
            solver = trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'sag'])

            params = {
                'C': trial.suggest_float('C', 0.001, 10, log=True),
                'penalty': penalty,
                'solver': solver,
                'max_iter': trial.suggest_int('max_iter', 100, 500, step=100),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
                'random_state': 123,
                'n_jobs': -1
            }
            model = LogisticRegression(**params)

        elif model_name == 'DecisionTree':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
                'random_state': 123
            }
            model = DecisionTreeClassifier(**params)

        elif model_name == 'RandomForest':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 250, step=50),
                'max_depth': trial.suggest_int('max_depth', 5, 25),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
                'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
                'random_state': 123,
                'n_jobs': -1
            }
            model = RandomForestClassifier(**params)

        elif model_name == 'XGBoost':
            params = {
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'n_estimators': trial.suggest_int('n_estimators', 50, 250, step=50),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 8),
                'gamma': trial.suggest_float('gamma', 0, 0.3),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 0.5, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 0.5, log=True),
                'random_state': 123,
                'eval_metric': 'logloss',
                'use_label_encoder': False
            }
            model = XGBClassifier(**params)

        elif model_name == 'LightGBM':
            params = {
                'num_leaves': trial.suggest_int('num_leaves', 20, 100),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'n_estimators': trial.suggest_int('n_estimators', 50, 250, step=50),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 20),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 0.5, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 0.5, log=True),
                'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.3),
                'random_state': 123,
                'verbose': -1,
                'n_jobs': -1
            }
            model = lgb.LGBMClassifier(**params)

        # Crear pipeline con manejo de errores
        try:
            pipeline = Pipeline_imb([
                ('preproc', prep),
                ('sampling', sampler),
                ('classifier', model)
            ])
        except Exception as e:
            # Fallback: usar solo SMOTE
            pipeline = Pipeline_imb([
                ('preproc', prep),
                ('sampling', SMOTE(sampling_strategy=0.7, random_state=123)),
                ('classifier', model)
            ])

        return pipeline

    def objective(trial):
        try:
            pipeline = create_pipeline(trial)

            scores = cross_val_score(
                pipeline,
                X_train,
                y_train,
                cv=cv,
                scoring='roc_auc',
                n_jobs=-1,
                error_score='raise'
            )

            return scores.mean()
        except Exception as e:
            # Si hay error, devolver un valor muy bajo
            return 0.0

    # Crear estudio con manejo de errores
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=123),
        study_name=f'{model_name}_optimization'
    )

    # Ejecutar optimización con try-except
    try:
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    except Exception as e:
        print(f"Error en optimización: {e}")

    # Verificar si hay trials exitosos
    if len(study.trials) > 0 and study.best_value > 0:
        best_pipeline = create_pipeline(study.best_trial)
        best_pipeline.fit(X_train, y_train)

        return {
            'study': study,
            'best_params': study.best_params,
            'best_value': study.best_value,
            'model': best_pipeline
        }
    else:
        # Modelo por defecto si falla la optimización
        print(f"Usando modelo por defecto para {model_name}")

        if model_name == 'LogisticRegression':
            model = LogisticRegression(random_state=123, max_iter=1000, n_jobs=-1)
        elif model_name == 'DecisionTree':
            model = DecisionTreeClassifier(random_state=123)
        elif model_name == 'RandomForest':
            model = RandomForestClassifier(random_state=123, n_jobs=-1)
        elif model_name == 'XGBoost':
            model = XGBClassifier(random_state=123, eval_metric='logloss', use_label_encoder=False)
        elif model_name == 'LightGBM':
            model = lgb.LGBMClassifier(random_state=123, verbose=-1, n_jobs=-1)

        pipeline_default = Pipeline_imb([
            ('preproc', prep),
            ('sampling', SMOTE(sampling_strategy=0.7, random_state=123)),
            ('classifier', model)
        ])

        pipeline_default.fit(X_train, y_train)

        # Calcular valor aproximado
        scores = cross_val_score(pipeline_default, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)

        return {
            'study': None,
            'best_params': {'default': 'default_model'},
            'best_value': scores.mean(),
            'model': pipeline_default
        }

PASO 6: FUNCION MODELO CON OPTUNA


### 9. OPTIMIZAR TODOS LOS MODELOS

In [ ]:
modelos = [
    'LogisticRegression',
    'DecisionTree',
    'RandomForest',
    'XGBoost',
    'LightGBM'
]

resultados_modelos = {}
modelos_entrenados = {}

for modelo in modelos:
    print(f"\n{'#'*70}")
    print(f"OPTIMIZANDO: {modelo}")
    print(f"{'#'*70}")

    resultados = optimize_model(
        model_name=modelo,
        X_train=X_train,
        y_train=y_train,
        cv=cv,
        n_trials=25  # Reducido para evitar problemas
    )

    resultados_modelos[modelo] = resultados
    modelos_entrenados[modelo] = resultados['model']

    print(f"\nMejor AUC-ROC: {resultados['best_value']:.4f}")
    if resultados['best_params'] and 'default' not in resultados['best_params']:
        print("Mejores parámetros:")
        for param, value in resultados['best_params'].items():
            print(f"  {param}: {value}")

### 10. EVALUACIÓN COMPARATIVA DE MODELOS

In [ ]:
print("PASO 7: EVALUACIÓN COMPARATIVA DE MODELOS")

# DataFrame para almacenar métricas
metricas_comparativas = []

for nombre_modelo, modelo in modelos_entrenados.items():
    print(f"\n{'*'*50}")
    print(f"Evaluando: {nombre_modelo}")
    print(f"{'*'*50}")

    # Predicciones
    y_pred_train = modelo.predict(X_train)
    y_pred_val = modelo.predict(X_val)
    y_pred_final = modelo.predict(X_final)

    y_proba_train = modelo.predict_proba(X_train)[:, 1]
    y_proba_val = modelo.predict_proba(X_val)[:, 1]
    y_proba_final = modelo.predict_proba(X_final)[:, 1]

    # Calcular métricas
    metricas = {
        'Modelo': nombre_modelo,
        'AUC_Train': roc_auc_score(y_train, y_proba_train),
        'AUC_Val': roc_auc_score(y_val, y_proba_val),
        'AUC_Final': roc_auc_score(y_final, y_proba_final),
        'Accuracy_Train': accuracy_score(y_train, y_pred_train),
        'Accuracy_Val': accuracy_score(y_val, y_pred_val),
        'Accuracy_Final': accuracy_score(y_final, y_pred_final),
        'F1_Score_Final': f1_score(y_final, y_pred_final),
        'Precision_Final': precision_score(y_final, y_pred_final),
        'Recall_Final': recall_score(y_final, y_pred_final)
    }

    metricas_comparativas.append(metricas)

    print(f"\nMÉTRICAS PARA {nombre_modelo}:")
    print(f"   AUC-ROC Final: {metricas['AUC_Final']:.4f}")
    print(f"   Accuracy Final: {metricas['Accuracy_Final']:.4f}")
    print(f"   F1-Score Final: {metricas['F1_Score_Final']:.4f}")

# Crear DataFrame comparativo
df_metricas = pd.DataFrame(metricas_comparativas)
df_metricas = df_metricas.sort_values('AUC_Final', ascending=False)

print("TABLA COMPARATIVA DE MODELOS")
display(df_metricas.round(4))


### 11. GRÁFICO ROC COMPARATIVO

In [ ]:
print("PASO 8: GRÁFICO ROC COMPARATIVO")

plt.figure(figsize=(12, 8))
colors = ['blue', 'red', 'green', 'purple', 'orange', 'brown']

for i, (nombre_modelo, modelo) in enumerate(modelos_entrenados.items()):
    y_proba = modelo.predict_proba(X_final)[:, 1]
    fpr, tpr, _ = roc_curve(y_final, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, color=colors[i % len(colors)], lw=2,
             label=f'{nombre_modelo} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Aleatorio')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
plt.title('Curvas ROC Comparativas - Datos de Prueba Final', fontsize=14, pad=20)
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 12. IDENTIFICAR MEJOR MODELO

In [ ]:
print("PASO 9: IDENTIFICACIÓN DEL MEJOR MODELO")

mejor_modelo_idx = df_metricas['AUC_Final'].idxmax()
mejor_modelo_nombre = df_metricas.loc[mejor_modelo_idx, 'Modelo']
mejor_modelo_auc = df_metricas.loc[mejor_modelo_idx, 'AUC_Final']

print(f"\nMEJOR MODELO: {mejor_modelo_nombre}")
print(f"   AUC-ROC Final: {mejor_modelo_auc:.4f}")

### 13. GUARDADO DE MODELOS

In [ ]:
print("PASO 10: GUARDADO DE MODELOS Y RESULTADOS")

# Crear carpeta para resultados
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
carpeta_resultados = f'{path_base}/resultados_comparativos_{timestamp}'
os.makedirs(carpeta_resultados, exist_ok=True)

# Guardar cada modelo
for nombre_modelo, modelo in modelos_entrenados.items():
    modelo_path = f'{carpeta_resultados}/modelo_{nombre_modelo}.joblib'
    joblib.dump(modelo, modelo_path)
    print(f" Modelo {nombre_modelo} guardado")

# Guardar métricas
metricas_path = f'{carpeta_resultados}/metricas_comparativas.csv'
df_metricas.to_csv(metricas_path, index=False)
print(f"Métricas guardadas")

# Guardar mejor modelo
mejor_modelo_path = f'{carpeta_resultados}/MEJOR_MODELO_{mejor_modelo_nombre}.joblib'
joblib.dump(modelos_entrenados[mejor_modelo_nombre], mejor_modelo_path)

### 14. RESUMEN FINAL

In [ ]:
print("RESUMEN FINAL")
print(f"\nResultados guardados en: {carpeta_resultados}")
print("\nRANKING DE MODELOS:")

for idx, row in df_metricas.iterrows():
    medalla = "->" if idx == mejor_modelo_idx else "🥈" if idx == 1 else "🥉" if idx == 2 else "   "
    print(f"{medalla} {row['Modelo']}: AUC = {row['AUC_Final']:.4f}")